# SLO-Aware LLM Request Triage: Full Training & Policy Comparison

This notebook runs the end-to-end training and evaluation pipeline from [`experiments/run_pipeline.py`](../experiments/run_pipeline.py) without duplicating code:
1. Prepares the pinned BurstGPT dataset (`train` / `validation` / `test` windows).
2. Trains both **our baseline RL policy (`dqn`)** and **our RL policy with output-length request ranking (`dqn_ranked`)**.
3. Reloads and verifies checkpoints, then evaluates **FCFS**, **BurstGPT Length-Aware**, **RL Policy (Repo DQN)**, and **RL Policy + Output-Length Ranking** on identical paired validation windows.

Use `CPU_TEST_MODE = True` for a fast local smoke test (subset of seeds, episodes, and windows), and set `CPU_TEST_MODE = False` for the full training run on Colab or local CPU.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Locate repository root (works locally from repo root or notebooks/ and on Google Colab)
repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent
elif Path("/content/llm-request-triage").exists():
    repo = Path("/content/llm-request-triage")

os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
if str(repo / "src") not in sys.path:
    sys.path.insert(0, str(repo / "src"))

if Path("/content").exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", ".[dev,rl]", "pandas>=2.0", "matplotlib>=3.7"],
        check=True,
    )

print(f"Working directory: {Path.cwd()}")

## 1. Configure and Run the Training + Comparison Loop

- **`CPU_TEST_MODE = True`**: Runs 1 seed (`7`), 2 episodes, 4 training windows, and 2 validation windows at 1 arrival rate so you can smoke-test the full loop locally in a few seconds.
- **`CPU_TEST_MODE = False`**: Runs the full configuration from `CONFIG_PATH` (3 seeds `[7, 17, 27]`, 24 episodes each, all 32 training windows and 8 validation windows across arrival rates).
- **`CONFIG_PATH`**: Defaults to `configs/week2_dqn.json` (standard RTX 5070 profile). You can also switch to `configs/training_kv_stress.json` to test under constrained KV-cache/batch capacity.

In [ ]:
from experiments.run_pipeline import run_training_loop

CPU_TEST_MODE = True  # Set to False for the full training run
CONFIG_PATH = "configs/week2_dqn.json"  # Or "configs/training_kv_stress.json"

results = run_training_loop(
    cpu_test_mode=CPU_TEST_MODE,
    include_ranked=True,
    config=CONFIG_PATH,
)

print(f"Artifacts saved to: {results['output_dir']}")

## 2. Four-Way Policy Comparison

Comparison of:
1. **FCFS** (`fcfs`)
2. **BurstGPT Length-Aware** (`length_aware`)
3. **RL Policy (Repo DQN)** (`dqn_seed*`)
4. **RL Policy + Output-Length Ranking** (`dqn_ranked_seed*`)

In [ ]:
import pandas as pd

comparison_df = pd.DataFrame(results["comparison_rows"])
summary_df = pd.DataFrame(results["summary_rows"])

print("=== 4-Way Policy Family Comparison ===")
display(comparison_df[[
    "policy_label",
    "policies",
    "paired_cases",
    "mean_slo_attainment",
    "best_seed_slo_attainment",
    "delta_vs_fcfs_pp",
    "mean_drop_rate",
    "mean_unfinished_rate",
    "mean_goodput_rps",
    "mean_p95_ttft_s",
]])

print("=== Per-Policy / Per-Seed Summary ===")
display(summary_df)

# Filter to constrained windows (windows where FCFS attainment < 0.90)
val_df = pd.DataFrame(results["validation_rows"])
fcfs_by_case = val_df[val_df["policy"] == "fcfs"].set_index(["window", "target_rps"])["slo_attainment"]
constrained_cases = fcfs_by_case[fcfs_by_case < 0.90].index
if len(constrained_cases) > 0:
    constrained_df = val_df[val_df.set_index(["window", "target_rps"]).index.isin(constrained_cases)]
    print(f"\n=== Constrained Windows Breakdown ({len(constrained_cases)} window-rate cases where FCFS < 90%) ===")
    display(
        constrained_df.pivot_table(
            index=["window", "target_rps"],
            columns="policy",
            values="slo_attainment",
        )
    )

## 3. Visualize Policy Comparison & Training Progression

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

plot_df = comparison_df.set_index("policy_label")
plot_df["mean_slo_attainment"].plot.bar(
    ax=axes[0], ylim=(0, 1.05), color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"],
    title="Mean SLO Attainment",
)
axes[0].set_ylabel("Attainment")

plot_df["mean_goodput_rps"].plot.bar(
    ax=axes[1], color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"],
    title="Mean Window Goodput (req/s)",
)
axes[1].set_ylabel("Goodput (req/s)")

plot_df["mean_drop_rate"].plot.bar(
    ax=axes[2], ylim=(0, 1.05), color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"],
    title="Mean Proactive Drop Rate",
)
axes[2].set_ylabel("Drop Rate")

for ax in axes:
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()